In [ ]:
%load_ext autoreload
%autoreload 2

from funkcje.funkcje import *
from funkcje.wskazniki_kredytowe import oblicz_wskazniki

In [25]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split, ParameterGrid, ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier


In [4]:
#Dane
df = pd.read_csv("zbior_1.csv")
#usuwam wybrane zmienne kategoryczne na podstawie tego co w eksploracja_danych.ipynb
X=df.iloc[:, :-1].drop(columns=["szczegolnaFormaPrawna_Symbol", "schemat_wsk_rzis","schemat_wsk_bilans"])
X=oblicz_wskazniki(X)
y=df.iloc[:, -1]

#Podzil na zbior treningowy (70%) i pozostale
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=2137,
    stratify=y
)

#Podzil na zbior walidacyjny (15%) i testowy (15%)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=2137,
    stratify=y_temp
)

# Najprostszy xgboost

In [7]:
#zmianna zmienych kategoryczeny na typ category i usniecie inf
categorical_columns = X_train.select_dtypes(include=['str', 'int']).columns
categorical_columns = categorical_columns.drop('wsk_liczba_dni_istnienia')

for col in categorical_columns:
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')
    X_test[col] = X_test[col].astype('category')

X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_valid = X_valid.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)


In [8]:
model = XGBClassifier(
    enable_categorical=True,
    tree_method="hist",
    random_state=2137
)

model.fit(X_train, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [10]:
X_valid = X_valid.copy()

categorical_columns = X_train.select_dtypes(
    include="category"
).columns

for col in categorical_columns:
    X_valid[col] = pd.Categorical(
        X_valid[col],
        categories=X_train[col].cat.categories
    )
    
predictions = model.predict_proba(X_valid)
roc_auc = roc_auc_score(y_valid, predictions[:, 1])

C:\Users\nortr\AppData\Local\Temp\ipykernel_78352\2714003286.py:8: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_valid[col] = pd.Categorical(


In [12]:
roc_auc

0.6395731707317073

# funkcja do czysczenia danych

In [20]:

class PrzygotowanieCech_xgb(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        kolumny_do_pominiecia=None,
        prog_korelacji=0.9,
        tylko_wskazniki=False,
        prog_nan=0.5,
        prog_zer=0.5,
        uwzglednij_kategoryczne=True,
    ):
        self.kolumny_do_pominiecia = kolumny_do_pominiecia
        self.prog_korelacji = prog_korelacji
        self.tylko_wskazniki = tylko_wskazniki
        self.prog_nan = prog_nan
        self.prog_zer = prog_zer
        self.uwzglednij_kategoryczne = uwzglednij_kategoryczne

    def _przygotuj_baze(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Dane wejściowe muszą być obiektem pandas.DataFrame.")

        pomin = self.kolumny_do_pominiecia or []
        brakujace = sorted(set(pomin) - set(X.columns))

        if brakujace:
            raise ValueError(
                f"Brak wymaganych kolumn do pominięcia: {brakujace}"
            )

        baza = X.drop(columns=pomin).copy()
        kolumny_poczatkowe = set(baza.columns)

        baza = oblicz_wskazniki(baza)

        if self.tylko_wskazniki:
            nowe_kolumny = [
                kolumna for kolumna in baza.columns
                if kolumna not in kolumny_poczatkowe
            ]
            baza = baza[nowe_kolumny]

        return baza

    @staticmethod
    def _kolumny_numeryczne(X):
        kolumny = X.select_dtypes(include=["float"]).columns.tolist()

        if (
            "wsk_liczba_dni_istnienia" in X.columns
            and "wsk_liczba_dni_istnienia" not in kolumny
        ):
            kolumny.append("wsk_liczba_dni_istnienia")

        return kolumny

    def fit(self, X, y=None):
        baza = self._przygotuj_baze(X)

        self.kolumny_numeryczne_ = self._kolumny_numeryczne(baza)

        self.kolumny_kategoryczne_ = (
            [
                col for col in baza.columns
                if col not in self.kolumny_numeryczne_
            ]
            if self.uwzglednij_kategoryczne
            else []
        )

        self.kategorie_ = {
            col: baza[col].dropna().unique().tolist()
            for col in self.kolumny_kategoryczne_
        }

        numeryczne = baza[self.kolumny_numeryczne_].copy()
        numeryczne = numeryczne.replace([np.inf, -np.inf], np.nan)

        maska_nan = numeryczne.isna().mean() <= self.prog_nan
        maska_zer = numeryczne.eq(0).mean() <= self.prog_zer
        maska_jakosci = maska_nan & maska_zer

        self.kolumny_zbyt_wielu_brakow_ = (
            numeryczne.columns[~maska_jakosci].tolist()
        )

        numeryczne = numeryczne.loc[:, maska_jakosci]

        if numeryczne.shape[1] > 0:
            po_korelacji = usuwanie_najbardziej_scorrelowanych(
                numeryczne,
                prog=self.prog_korelacji
            )
            self.kolumny_po_korelacji_ = po_korelacji.columns.tolist()
        else:
            self.kolumny_po_korelacji_ = []

        self.kolumny_wynikowe_ = (
            self.kolumny_po_korelacji_
            + self.kolumny_kategoryczne_
        )

        if not self.kolumny_wynikowe_:
            raise ValueError("Nie pozostały żadne cechy po filtrowaniu.")

        return self

    def transform(self, X):
        baza = self._przygotuj_baze(X)

        numeryczne = baza.reindex(
            columns=self.kolumny_po_korelacji_
        ).copy()

        numeryczne = numeryczne.replace(
            [np.inf, -np.inf],
            np.nan
        )

        if not self.uwzglednij_kategoryczne:
            return numeryczne

        kategoryczne = pd.DataFrame(index=baza.index)

        for col in self.kolumny_kategoryczne_:
            wartosci = (
                baza[col]
                if col in baza.columns
                else pd.Series(np.nan, index=baza.index)
            )

            kategoryczne[col] = pd.Categorical(
                wartosci,
                categories=self.kategorie_[col]
            )

        wynik = pd.concat(
            [numeryczne, kategoryczne],
            axis=1
        )

        return wynik.reindex(
            columns=self.kolumny_wynikowe_
        )


# Pipeline i dopasownie hiperparametrow

In [22]:
pipeline = Pipeline([
    ("przygotowanie_cech", PrzygotowanieCech_xgb()),
    ("classifier", XGBClassifier(
        enable_categorical=True,
        tree_method="hist",
        random_state=2137)
    )
])

pipeline.fit(X_train, y_train)

predictions = pipeline.predict_proba(X_valid)[:, 1]
print(f"AUC: {roc_auc_score(y_valid, predictions):.4f}")

AUC: 0.6599


In [23]:

from sklearn.model_selection import ParameterGrid
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
import pandas as pd

param_grid = {
    # Przygotowanie cech
    "przygotowanie_cech__uwzglednij_kategoryczne": [True, False],
    "przygotowanie_cech__prog_nan": [0.25, 0.35, 0.45, 0.55, 0.65],
    "przygotowanie_cech__prog_zer": [0.3],
    "przygotowanie_cech__prog_korelacji": [0.7, 0.8, 0.9, 1.0],

    # XGBoost
    "classifier__n_estimators": [500, 800, 1100, 1500, 2000],
    "classifier__max_depth": [2, 3, 4, 5, 7],
    "classifier__learning_rate": [0.05],
    "classifier__min_child_weight": [3],
}

wyniki = []
najlepszy_auc = -float("inf")
najlepszy_model = None
najlepsze_parametry = None

kombinacje = list(ParameterGrid(param_grid))
print(f"Liczba kombinacji: {len(kombinacje)}")

for i, parametry in enumerate(kombinacje, start=1):
    model = clone(pipeline)
    model.set_params(**parametry)

    try:
        model.fit(X_train, y_train)

        prawdopodobienstwo = model.predict_proba(X_valid)[:, 1]
        auc = roc_auc_score(y_valid, prawdopodobienstwo)

        wyniki.append({**parametry, "auc": auc})

        if auc > najlepszy_auc:
            najlepszy_auc = auc
            najlepszy_model = model
            najlepsze_parametry = parametry

        print(f"[{i}/{len(kombinacje)}] AUC: {auc:.4f}")

    except (ValueError, TypeError) as e:
        print(f"[{i}/{len(kombinacje)}] Błąd: {e}")

wyniki_df = (
    pd.DataFrame(wyniki)
    .sort_values("auc", ascending=False)
    .reset_index(drop=True)
)

print("\nNajlepsze parametry:")
print(najlepsze_parametry)
print(f"\nNajlepszy ROC AUC: {najlepszy_auc:.4f}")

display(wyniki_df)


Liczba kombinacji: 1000
[1/1000] AUC: 0.6156
[2/1000] AUC: 0.8310
[3/1000] AUC: 0.6156
[4/1000] AUC: 0.8310
[5/1000] AUC: 0.6183
[6/1000] AUC: 0.8205
[7/1000] AUC: 0.6183
[8/1000] AUC: 0.8205
[9/1000] AUC: 0.6180
[10/1000] AUC: 0.8248
[11/1000] AUC: 0.6190
[12/1000] AUC: 0.8148
[13/1000] AUC: 0.6190
[14/1000] AUC: 0.8148
[15/1000] AUC: 0.6188
[16/1000] AUC: 0.8143
[17/1000] AUC: 0.6188
[18/1000] AUC: 0.8143
[19/1000] AUC: 0.6188
[20/1000] AUC: 0.8095
[21/1000] AUC: 0.6217
[22/1000] AUC: 0.8113
[23/1000] AUC: 0.6217
[24/1000] AUC: 0.8113
[25/1000] AUC: 0.6177
[26/1000] AUC: 0.8132
[27/1000] AUC: 0.6177
[28/1000] AUC: 0.8132
[29/1000] AUC: 0.6245
[30/1000] AUC: 0.8109
[31/1000] AUC: 0.6426
[32/1000] AUC: 0.8138
[33/1000] AUC: 0.6358
[34/1000] AUC: 0.8149
[35/1000] AUC: 0.6388
[36/1000] AUC: 0.8090
[37/1000] AUC: 0.6388
[38/1000] AUC: 0.8090
[39/1000] AUC: 0.6388
[40/1000] AUC: 0.8077
[41/1000] AUC: 0.6043
[42/1000] AUC: 0.8199
[43/1000] AUC: 0.6043
[44/1000] AUC: 0.8199
[45/1000] AUC: 0.

,classifier__learning_rate,classifier__max_depth,classifier__min_child_weight,classifier__n_estimators,przygotowanie_cech__prog_korelacji,przygotowanie_cech__prog_nan,przygotowanie_cech__prog_zer,przygotowanie_cech__uwzglednij_kategoryczne,auc
0,0.05,2,3,500,0.7,0.25,0.3,False,0.831037
1,0.05,2,3,500,0.7,0.35,0.3,False,0.831037
2,0.05,2,3,500,0.7,0.65,0.3,False,0.824817
3,0.05,2,3,500,0.7,0.45,0.3,False,0.820549
4,0.05,2,3,500,0.7,0.55,0.3,False,0.820549
...,...,...,...,...,...,...,...,...,...
995,0.05,2,3,800,0.7,0.25,0.3,True,0.604329
996,0.05,2,3,1100,0.7,0.25,0.3,True,0.602927
997,0.05,2,3,1100,0.7,0.35,0.3,True,0.602927
998,0.05,2,3,1100,0.7,0.45,0.3,True,0.602622


In [27]:
param_grid = {
    # Przygotowanie cech – dotychczas najlepsze wartości
    "przygotowanie_cech__prog_korelacji": [0.7, 0.9],
    "przygotowanie_cech__prog_nan": [0.25, 0.5],
    "przygotowanie_cech__prog_zer": [0.3],
    "przygotowanie_cech__uwzglednij_kategoryczne": [False],

    # XGBoost – liczba i głębokość drzew
    "classifier__n_estimators": [300, 500, 800, 1500],
    "classifier__max_depth": [2, 3],
    "classifier__learning_rate": [0.03, 0.05],
    "classifier__min_child_weight": [1, 3],

    # XGBoost – losowość i próbkowanie
    "classifier__subsample": [0.7, 1.0],
    "classifier__colsample_bytree": [0.7,  1.0],

    # XGBoost – regularyzacja
    "classifier__reg_lambda": [1, 5, 10],
    "classifier__reg_alpha": [0, 0.1, 1],
    "classifier__gamma": [0, 1, 3],
}

wyniki = []
najlepszy_auc = -float("inf")
najlepszy_model = None
najlepsze_parametry = None

kombinacje = list(ParameterSampler(
    param_distributions=param_grid,
    n_iter=1000,
    random_state=2137
))
print(f"Liczba kombinacji: {len(kombinacje)}")

for i, parametry in enumerate(kombinacje, start=1):
    model = clone(pipeline)
    model.set_params(**parametry)

    try:
        model.fit(X_train, y_train)

        prawdopodobienstwo = model.predict_proba(X_valid)[:, 1]
        auc = roc_auc_score(y_valid, prawdopodobienstwo)

        wyniki.append({**parametry, "auc": auc})

        if auc > najlepszy_auc:
            najlepszy_auc = auc
            najlepszy_model = model
            najlepsze_parametry = parametry

        print(f"[{i}/{len(kombinacje)}] AUC: {auc:.4f}")

    except (ValueError, TypeError) as e:
        print(f"[{i}/{len(kombinacje)}] Błąd: {e}")

wyniki_df = (
    pd.DataFrame(wyniki)
    .sort_values("auc", ascending=False)
    .reset_index(drop=True)
)

print("\nNajlepsze parametry:")
print(najlepsze_parametry)
print(f"\nNajlepszy ROC AUC: {najlepszy_auc:.4f}")

display(wyniki_df)

Liczba kombinacji: 1000
[1/1000] AUC: 0.8248
[2/1000] AUC: 0.8122
[3/1000] AUC: 0.8260
[4/1000] AUC: 0.8296
[5/1000] AUC: 0.8261
[6/1000] AUC: 0.8084
[7/1000] AUC: 0.8040
[8/1000] AUC: 0.8345
[9/1000] AUC: 0.8285
[10/1000] AUC: 0.8374
[11/1000] AUC: 0.8367
[12/1000] AUC: 0.7660
[13/1000] AUC: 0.8071
[14/1000] AUC: 0.8116
[15/1000] AUC: 0.8003
[16/1000] AUC: 0.8235
[17/1000] AUC: 0.7749
[18/1000] AUC: 0.8340
[19/1000] AUC: 0.8259
[20/1000] AUC: 0.8315
[21/1000] AUC: 0.8276
[22/1000] AUC: 0.7901
[23/1000] AUC: 0.8287
[24/1000] AUC: 0.7846
[25/1000] AUC: 0.8004
[26/1000] AUC: 0.8249
[27/1000] AUC: 0.8167
[28/1000] AUC: 0.8168
[29/1000] AUC: 0.8321
[30/1000] AUC: 0.8282
[31/1000] AUC: 0.8019
[32/1000] AUC: 0.8317
[33/1000] AUC: 0.8316
[34/1000] AUC: 0.8250
[35/1000] AUC: 0.8153
[36/1000] AUC: 0.8215
[37/1000] AUC: 0.7967
[38/1000] AUC: 0.8216
[39/1000] AUC: 0.8357
[40/1000] AUC: 0.7621
[41/1000] AUC: 0.8255
[42/1000] AUC: 0.8072
[43/1000] AUC: 0.8239
[44/1000] AUC: 0.8321
[45/1000] AUC: 0.

,przygotowanie_cech__uwzglednij_kategoryczne,przygotowanie_cech__prog_zer,przygotowanie_cech__prog_nan,przygotowanie_cech__prog_korelacji,classifier__subsample,classifier__reg_lambda,classifier__reg_alpha,classifier__n_estimators,classifier__min_child_weight,classifier__max_depth,classifier__learning_rate,classifier__gamma,classifier__colsample_bytree,auc
0,False,0.3,0.25,0.7,1.0,5,1.0,1500,3,2,0.03,1,1.0,0.841402
1,False,0.3,0.25,0.7,1.0,10,0.1,300,3,2,0.03,0,1.0,0.840122
2,False,0.3,0.25,0.7,1.0,10,1.0,300,3,2,0.03,0,0.7,0.840000
3,False,0.3,0.50,0.7,1.0,5,1.0,300,3,2,0.03,1,0.7,0.839695
4,False,0.3,0.50,0.7,1.0,10,0.1,300,3,2,0.03,0,1.0,0.839573
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,False,0.3,0.50,0.9,0.7,5,0.1,1500,3,3,0.05,0,1.0,0.754878
996,False,0.3,0.50,0.9,0.7,1,0.0,1500,1,3,0.05,0,0.7,0.753902
997,False,0.3,0.50,0.7,0.7,5,0.1,1500,3,3,0.05,0,1.0,0.753232
998,False,0.3,0.50,0.9,0.7,5,1.0,1500,3,3,0.05,0,0.7,0.751951


In [29]:
param_grid = {
    # Przygotowanie cech – najlepsze wartości
    "przygotowanie_cech__uwzglednij_kategoryczne": [False],
    "przygotowanie_cech__prog_zer": [0.3],
    "przygotowanie_cech__prog_nan": [0.25],
    "przygotowanie_cech__prog_korelacji": [0.7],

    # XGBoost – struktura
    "classifier__n_estimators": [100, 200, 300, 400, 500],
    "classifier__max_depth": [2],
    "classifier__min_child_weight": [1, 3],

    # XGBoost – szybkość uczenia
    "classifier__learning_rate": [0.02, 0.03],

    # XGBoost – regularyzacja
    "classifier__reg_alpha": [0.5, 1, 2, 3, 5],
    "classifier__reg_lambda": [3, 5, 10, 15, 20],
    "classifier__gamma": [0, 1],

    # XGBoost – próbkowanie
    "classifier__subsample": [1.0],
    "classifier__colsample_bytree": [1.0],
}

kombinacje = list(ParameterGrid(param_grid))

wyniki = []
najlepszy_auc = -float("inf")
najlepszy_model = None
najlepsze_parametry = None

kombinacje = list(ParameterGrid(param_grid))
print(f"Liczba kombinacji: {len(kombinacje)}")

for i, parametry in enumerate(kombinacje, start=1):
    model = clone(pipeline)
    model.set_params(**parametry)

    try:
        model.fit(X_train, y_train)

        prawdopodobienstwo = model.predict_proba(X_valid)[:, 1]
        auc = roc_auc_score(y_valid, prawdopodobienstwo)

        wyniki.append({**parametry, "auc": auc})

        if auc > najlepszy_auc:
            najlepszy_auc = auc
            najlepszy_model = model
            najlepsze_parametry = parametry

        print(f"[{i}/{len(kombinacje)}] AUC: {auc:.4f}")

    except (ValueError, TypeError) as e:
        print(f"[{i}/{len(kombinacje)}] Błąd: {e}")

wyniki_df = (
    pd.DataFrame(wyniki)
    .sort_values("auc", ascending=False)
    .reset_index(drop=True)
)

print("\nNajlepsze parametry:")
print(najlepsze_parametry)
print(f"\nNajlepszy ROC AUC: {najlepszy_auc:.4f}")

display(wyniki_df)

Liczba kombinacji: 1000
[1/1000] AUC: 0.8152
[2/1000] AUC: 0.8157
[3/1000] AUC: 0.8117
[4/1000] AUC: 0.8088
[5/1000] AUC: 0.8088
[6/1000] AUC: 0.8141
[7/1000] AUC: 0.8155
[8/1000] AUC: 0.8138
[9/1000] AUC: 0.8088
[10/1000] AUC: 0.8079
[11/1000] AUC: 0.8162
[12/1000] AUC: 0.8159
[13/1000] AUC: 0.8130
[14/1000] AUC: 0.8100
[15/1000] AUC: 0.8094
[16/1000] AUC: 0.8168
[17/1000] AUC: 0.8159
[18/1000] AUC: 0.8131
[19/1000] AUC: 0.8098
[20/1000] AUC: 0.8073
[21/1000] AUC: 0.8188
[22/1000] AUC: 0.8179
[23/1000] AUC: 0.8137
[24/1000] AUC: 0.8094
[25/1000] AUC: 0.8050
[26/1000] AUC: 0.8333
[27/1000] AUC: 0.8313
[28/1000] AUC: 0.8281
[29/1000] AUC: 0.8250
[30/1000] AUC: 0.8227
[31/1000] AUC: 0.8304
[32/1000] AUC: 0.8317
[33/1000] AUC: 0.8276
[34/1000] AUC: 0.8244
[35/1000] AUC: 0.8224
[36/1000] AUC: 0.8322
[37/1000] AUC: 0.8295
[38/1000] AUC: 0.8256
[39/1000] AUC: 0.8253
[40/1000] AUC: 0.8232
[41/1000] AUC: 0.8312
[42/1000] AUC: 0.8282
[43/1000] AUC: 0.8251
[44/1000] AUC: 0.8228
[45/1000] AUC: 0.

,classifier__colsample_bytree,classifier__gamma,classifier__learning_rate,classifier__max_depth,classifier__min_child_weight,classifier__n_estimators,classifier__reg_alpha,classifier__reg_lambda,classifier__subsample,przygotowanie_cech__prog_korelacji,przygotowanie_cech__prog_nan,przygotowanie_cech__prog_zer,przygotowanie_cech__uwzglednij_kategoryczne,auc
0,1.0,0,0.02,2,3,400,0.5,5,1.0,0.7,0.25,0.3,False,0.843476
1,1.0,0,0.03,2,3,400,3.0,5,1.0,0.7,0.25,0.3,False,0.842988
2,1.0,0,0.02,2,3,400,1.0,5,1.0,0.7,0.25,0.3,False,0.842927
3,1.0,0,0.03,2,3,300,2.0,5,1.0,0.7,0.25,0.3,False,0.842866
4,1.0,1,0.02,2,3,400,1.0,5,1.0,0.7,0.25,0.3,False,0.842683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1.0,1,0.02,2,1,100,3.0,20,1.0,0.7,0.25,0.3,False,0.807287
996,1.0,1,0.02,2,3,100,5.0,20,1.0,0.7,0.25,0.3,False,0.805000
997,1.0,1,0.02,2,1,100,5.0,20,1.0,0.7,0.25,0.3,False,0.805000
998,1.0,0,0.02,2,3,100,5.0,20,1.0,0.7,0.25,0.3,False,0.805000


In [36]:
param_grid = {
    # Przygotowanie cech – najlepsze wartości
    "przygotowanie_cech__uwzglednij_kategoryczne": [False],
    "przygotowanie_cech__prog_zer": [0.3],
    "przygotowanie_cech__prog_nan": [0.25],
    "przygotowanie_cech__prog_korelacji": [0.7],

    # XGBoost – struktura drzew
    "classifier__n_estimators": [200, 300, 400, 450],
    "classifier__max_depth": [2],
    "classifier__min_child_weight": [3],

    # XGBoost – szybkość uczenia
    "classifier__learning_rate": [0.02, 0.03],

    # XGBoost – regularyzacja
    "classifier__reg_alpha": [0.5, 1, 2, 3, 5],
    "classifier__reg_lambda": [2, 3, 5, 7, 10],
    "classifier__gamma": [0, 1],

    # XGBoost – próbkowanie
    "classifier__subsample": [1.0],
    "classifier__colsample_bytree": [1.0],
}
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2137
)

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=2,
    verbose=2,
    refit=True,
    return_train_score=True,
    error_score="raise"
)

search.fit(X_train, y_train)

print("Najlepsze parametry:")
print(search.best_params_)

print(f"\nNajlepszy CV ROC AUC: {search.best_score_:.6f}")

najlepszy_model = search.best_estimator_

Fitting 5 folds for each of 400 candidates, totalling 2000 fits
Najlepsze parametry:
{'classifier__colsample_bytree': 1.0, 'classifier__gamma': 1, 'classifier__learning_rate': 0.03, 'classifier__max_depth': 2, 'classifier__min_child_weight': 3, 'classifier__n_estimators': 400, 'classifier__reg_alpha': 1, 'classifier__reg_lambda': 10, 'classifier__subsample': 1.0, 'przygotowanie_cech__prog_korelacji': 0.7, 'przygotowanie_cech__prog_nan': 0.25, 'przygotowanie_cech__prog_zer': 0.3, 'przygotowanie_cech__uwzglednij_kategoryczne': False}

Najlepszy CV ROC AUC: 0.807032


In [40]:
najlepszy_model.predict_proba(X_test)
roc_auc = roc_auc_score(y_test, najlepszy_model.predict_proba(X_test)[:, 1])
print(f"ROC AUC na zbiorze testowym: {roc_auc:.6f}")

ROC AUC na zbiorze testowym: 0.795808
